# Track A · Google Colab
런타임 유형을 GPU로 바꾼 뒤 순서대로 실행합니다. Smoke와 정식 실험은 서로 다른 ID를 사용합니다.

In [ ]:
import torch
import subprocess
import sys
from pathlib import Path

def run(*args):
    subprocess.run([str(arg) for arg in args], check=True)

run("nvidia-smi")
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/Deep_learning")
results_dir = drive_root / "results"
checkpoints_dir = drive_root / "checkpoints"
for directory in (results_dir, checkpoints_dir):
    directory.mkdir(parents=True, exist_ok=True)


공개 저장소는 URL을 입력합니다. 비공개 저장소는 인증된 로컬 환경에서 `git archive --format=zip -o Deep_learning.zip HEAD`로 만든 ZIP을 Drive에 저장해 경로를 지정합니다. 정식 재현에는 커밋 SHA를 기록합니다.

In [ ]:
import os
import shutil
from zipfile import ZipFile

repo_url = "https://github.com/YOUR_ORG/Deep_learning.git"
commit = ""  # 공개 저장소에서 고정할 커밋 SHA
private_zip = ""  # 예: /content/drive/MyDrive/Deep_learning/Deep_learning.zip
repo = Path("/content/Deep_learning")
if private_zip:
    with ZipFile(private_zip) as archive:
        archive_commit = archive.comment.decode("ascii")
    assert len(archive_commit) == 40 and all(c in "0123456789abcdef" for c in archive_commit), "Use git archive"
    if commit:
        assert commit == archive_commit, "ZIP commit does not match"
    marker = repo / ".archive-commit"
    if not repo.exists():
        shutil.unpack_archive(private_zip, repo)
        marker.write_text(archive_commit)
    assert marker.is_file() and marker.read_text() == archive_commit, "Use a fresh Colab runtime to change archive"
    os.environ["TRACK_A_ARCHIVE_COMMIT"] = archive_commit
    print(archive_commit)
else:
    assert "YOUR_ORG" not in repo_url, "GitHub URL을 입력하세요."
    if not repo.exists():
        run("git", "clone", repo_url, repo)
    assert (repo / ".git").exists(), "Use a fresh Colab runtime to change source"
    if commit:
        run("git", "-C", repo, "fetch", "origin")
        run("git", "-C", repo, "checkout", "--detach", commit)
    run("git", "-C", repo, "rev-parse", "HEAD")
os.chdir(repo)
assert Path("train.py").is_file(), "코드 파일이 ZIP 또는 저장소 루트에 있어야 합니다."
run(sys.executable, "-m", "pip", "install", "-r", "requirements.txt")


In [ ]:
run(sys.executable, "dataset.py")
from model import build_model, parameter_count
b0 = build_model("B0").cuda().eval()
with torch.no_grad():
    output = b0(torch.zeros(2, 3, 32, 32, device="cuda"))
assert output.shape == (2, 10)
print(output.shape, parameter_count(b0))
del b0, output


In [ ]:
smoke_id = "colab_smoke_b0"
resume = ["--resume"] if (checkpoints_dir / smoke_id / "last.pt").exists() else []
run(sys.executable, "train.py", "--device", "cuda", "--model", "B0", "--ratio", 10,
    "--seed", 0, "--epochs", 1, "--run-id", smoke_id,
    "--results-dir", results_dir, "--checkpoints-dir", checkpoints_dir, *resume)
print(results_dir / smoke_id, checkpoints_dir / smoke_id)


다음 셀의 ratios를 팀 결정과 맞춥니다. 생성된 CSV에서 본인 담당 실행 ID를 하나 선택합니다. 탐색을 마치면 PLAN.md 순서에 따라 select, core, ablation을 실행합니다.

In [ ]:
ratios = [100, 25, 10]
paths = ["--results-dir", results_dir, "--checkpoints-dir", checkpoints_dir]
tune_manifest = results_dir / "tune_plan.csv"
if not tune_manifest.exists():
    run(sys.executable, "experiments.py", "tune", "--ratios", *ratios, *paths)
print(tune_manifest)


In [ ]:
manifest = tune_manifest  # 단계에 따라 core_plan.csv 또는 ablation_plan.csv 지정
run_id = ""  # 실험표의 실제 ID
resume = False
if run_id:
    run(sys.executable, "experiments.py", "run", "--output", manifest, "--run-id", run_id,
        "--device", "cuda", *(["--resume"] if resume else []))
    print(results_dir / run_id, checkpoints_dir / run_id)


팀이 비교 조건과 Ablation 대상을 고정한 후 최종 평가를 실행합니다. Git 인증 정보는 노트북, 결과 또는 명령 기록에 넣지 않습니다.

In [ ]:
final_id = ""
if final_id:
    run(sys.executable, "evaluate.py", checkpoints_dir / final_id / "last.pt",
        "--device", "cuda", "--results-dir", results_dir, "--flops")
